# Pipeline Hán - Việt: Tự Động Dóng Hàng & Huấn Luyện Máy Dịch trên Kaggle GPU (Zero-Setup)

Notebook này tự động thực hiện toàn bộ quy trình từ **Dóng hàng câu** đến **Huấn luyện máy dịch** bằng cách clone trực tiếp mã nguồn và dữ liệu từ GitHub của bạn. Bạn không cần tải dữ liệu hay tạo Dataset thủ công trên Kaggle.

## Hướng dẫn chạy nhanh:
1. Hãy push toàn bộ nhánh `features/mapping-translation` hiện tại ở local lên GitHub của bạn.
2. Tạo một Kaggle Notebook mới, thiết lập **Accelerator** là **GPU T4 x2** hoặc **GPU P100** ở bảng cài đặt bên phải.
3. Thay thế URL GitHub của bạn vào ô code bên dưới và nhấn **Run All**.

## Bước 1: Clone/Update Repository từ GitHub và chuyển thư mục làm việc

In [ ]:
# THAY THẾ URL GITHUB REPO CỦA BẠN VÀO ĐÂY
GITHUB_REPO_URL = "https://github.com/YOUR_GITHUB_USERNAME/SinoNom-NLP.git"

import os
repo_name = GITHUB_REPO_URL.split("/")[-1].replace(".git", "")

if not os.path.exists(repo_name):
    print(f"Cloning repository from {GITHUB_REPO_URL}...")
    # Clone nhánh features/mapping-translation chứa toàn bộ dữ liệu và code mới
    !git clone -b features/mapping-translation {GITHUB_REPO_URL}
else:
    print("Repository already exists. Fetching and updating to the latest commit...")
    %cd {repo_name}
    !git fetch --all
    !git reset --hard origin/features/mapping-translation
    %cd ..

# Chuyển thư mục làm việc vào trong repo
%cd {repo_name}
!ls -la

## Bước 2: Cài đặt các thư viện cần thiết

In [ ]:
# Cài đặt các thư viện phục vụ cho việc dóng hàng và huấn luyện dịch máy
!pip install -q sentence-transformers pandas openpyxl
!pip install -q transformers[torch] datasets evaluate sacrebleu accelerate tensorboard
!pip install -q bertalign simalign bitsandbytes

## Bước 2.5: Phân Tích & Khám Phá Dữ Liệu (EDA) trước khi Dóng Hàng

Trước khi thực hiện dóng hàng, ta cần chạy các phân tích thống kê cơ bản trên dataset mới để hiểu rõ phân phối độ dài câu, tỉ lệ Hán-Việt, và kiểm tra chất lượng dữ liệu.

In [ ]:
# Cell 1: Định nghĩa các nhóm mapping và kiểm tra file tồn tại
import os
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

SINO_DIR = 'dataset/MAPPING/sino_extract'
VIET_DIR = 'dataset/MAPPING/vietnam_extract/csv'

MAPPING_GROUPS = [
    ('Q01', ['q1_sentences.csv'], 'q01.csv'),
    ('Q02-04', ['q2_sentences.csv', 'q3_sentences.csv', 'q4_sentences.csv'], 'q2_3_4.csv'),
    ('Q05', ['q5_sentences.csv'], 'q05.csv'),
    ('Q06', ['q6_sentences.csv'], 'q6.csv'),
    ('Q07-08', ['q7_sentences.csv', 'q8_sentences.csv'], 'q07_08.csv'),
    ('Q09', ['q9_sentences.csv'], 'q09.csv'),
    ('Q10-11', ['q10_11_sentences.csv'], 'q10_11.csv'),
    ('Q12', ['q12_sentences.csv'], 'q12.csv'),
    ('Q13', ['q13_sentences.csv'], 'q13.csv'),
    ('Q14-15', ['q14_sentences.csv', 'q15_sentences.csv'], 'q14_15.csv'),
    ('Q16-17', ['q16_17_sentences.csv'], 'q16_17.csv'),
]

print(f"Loaded {len(MAPPING_GROUPS)} mapping groups successfully.")

In [ ]:
# Cell 2: Thống kê số câu và tỷ lệ giữa Hán và Việt
def count_valid_viet(df):
    heading_kws = [
        r'^quyển\s+', r'^tỉnh\s+', r'^phủ\s+', r'^đại\s*-?\s*nam',
        r'^dựng\s+đặt', r'^phân\s+dã', r'^phần\s+dã', r'^khí\s+hậu',
        r'^thành\s*-?\s*trì', r'^tử\s*-?\s*chí', r'^sông\s+núi',
        r'^danh\s+lam', r'^cổ\s+tự', r'^sản\s+vật', r'^nhân\s+vật',
    ]
    def is_valid(s):
        if not isinstance(s, str) or len(s.strip()) < 6:
            return False
        words = s.strip().split()
        uc = [w for w in words if w.isupper() or not w.isalpha()]
        if len(uc)/len(words) > 0.8 and len(s.strip()) < 45:
            return False
        sl = s.strip().lower()
        for kw in heading_kws:
            if re.search(kw, sl): return False
        return True
    return df['sentence'].apply(is_valid).sum()

rows = []
for group, sino_files, viet_file in MAPPING_GROUPS:
    n_han = 0
    for sf in sino_files:
        p = os.path.join(SINO_DIR, sf)
        if os.path.exists(p):
            sep = ';' if ';' in open(p, encoding='utf-8').readline() else ','
            n_han += len(pd.read_csv(p, sep=sep))
    vp = os.path.join(VIET_DIR, viet_file)
    n_viet_raw, n_viet_valid = 0, 0
    if os.path.exists(vp):
        dfv = pd.read_csv(vp)
        n_viet_raw = len(dfv)
        n_viet_valid = count_valid_viet(dfv)
    ratio = n_han / n_viet_valid if n_viet_valid > 0 else 0
    rows.append({
        'Nhóm': group,
        'Hán (câu)': n_han,
        'Việt (raw)': n_viet_raw,
        'Việt (valid)': n_viet_valid,
        'Tỷ lệ Hán:Việt': round(ratio, 2)
    })

df_overview = pd.DataFrame(rows)
display(df_overview)
print(f"Tổng Hán: {df_overview['Hán (câu)'].sum():,} câu")
print(f"Tổng Việt (hợp lệ): {df_overview['Việt (valid)'].sum():,} câu")

In [ ]:
# Cell 3: Trực quan hóa số lượng và phân phối độ dài câu
all_han_lens, all_viet_lens = [], []
for group, sino_files, viet_file in MAPPING_GROUPS:
    for sf in sino_files:
        p = os.path.join(SINO_DIR, sf)
        if os.path.exists(p):
            sep = ';' if ';' in open(p, encoding='utf-8').readline() else ','
            df = pd.read_csv(p, sep=sep)
            all_han_lens.extend(df['sentence'].dropna().apply(lambda x: len(str(x).strip())).tolist())
    vp = os.path.join(VIET_DIR, viet_file)
    if os.path.exists(vp):
        dfv = pd.read_csv(vp)
        all_viet_lens.extend(dfv['sentence'].dropna().apply(lambda x: len(str(x).strip().split())).tolist())

plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.hist(np.clip(all_han_lens, 0, 150), bins=30, color='skyblue', edgecolor='black')
plt.title('Độ dài câu Hán (số ký tự)')
plt.xlabel('Độ dài (ký tự)')
plt.ylabel('Số câu')

plt.subplot(1, 2, 2)
plt.hist(np.clip(all_viet_lens, 0, 80), bins=30, color='lightcoral', edgecolor='black')
plt.title('Độ dài câu Việt (số từ)')
plt.xlabel('Độ dài (từ)')
plt.ylabel('Số câu')

plt.tight_layout()
plt.show()

## Bước 3: Phase 1 — Chạy Ensemble Alignment (Không dùng Qwen)

Chúng ta sẽ chạy mô hình dóng hàng kết hợp (Ensemble Aligner) gồm LaBSE, Vecalign, BERTAlign, và SimAlign để tạo ra kết quả dóng hàng ban đầu mà chưa lọc qua Qwen.

**Lưu ý về Chế độ Thử nghiệm (Dev Mode):**
* Lệnh dưới đây mặc định bật cờ `--dev` để chạy thử nghiệm nhanh trên **Quyển 1**.
* Để chạy dóng hàng trên toàn bộ 17 quyển, hãy xóa cờ `--dev` đi.

In [ ]:
# Chạy Phase 1: Ensemble Alignment
!python run_mapping.py \
    --aligner ensemble \
    --sino_dir dataset/MAPPING/sino_extract \
    --viet_dir dataset/MAPPING/vietnam_extract/csv \
    --output_dir output \
    --work_code HVB_001 \
    --device cuda \
    --dev

### Kiểm tra trực quan kết quả Phase 1

In [ ]:
import glob
import pandas as pd
from IPython.display import display

tsv_files = glob.glob("output/HVB_001/**/*.tsv", recursive=True)
if tsv_files:
    sample_file = sorted(tsv_files)[0]
    print(f"Kết quả Phase 1 (File: {sample_file}):\n")
    df_sample = pd.read_csv(sample_file, sep="\t")
    pd.set_option('display.max_colwidth', None)
    display(df_sample.head(25))
    nan_count = df_sample['han_sentence'].isna().sum() + df_sample['viet_sentence'].isna().sum()
    print(f"\nTổng số dòng: {len(df_sample)} | Dòng NaN (unmatched): {nan_count}")
else:
    print("Chưa tìm thấy kết quả dóng hàng.")

### Tải về kết quả Phase 1

In [ ]:
!rm -f output_phase1.zip
!zip -q -r output_phase1.zip output/
print("Đã nén xong! Hãy tải file 'output_phase1.zip' ở cột bên phải Kaggle để kiểm tra.")

## Bước 3.5: Phase 2 — Qwen LLM Verification (Lọc & Tách lỗi)

Qwen2.5-7B-Instruct nạp ở chế độ 4-bit, duyệt qua các cặp có độ tin cậy thấp và tách rời chúng ra.

In [ ]:
# Chạy Phase 2: Qwen Verification
!python run_mapping.py \
    --aligner ensemble \
    --qwen \
    --sino_dir dataset/MAPPING/sino_extract \
    --viet_dir dataset/MAPPING/vietnam_extract/csv \
    --output_dir output \
    --work_code HVB_001 \
    --device cuda \
    --dev

### Kiểm tra kết quả sau Phase 2

In [ ]:
tsv_files = glob.glob("output/HVB_001/**/*.tsv", recursive=True)
if tsv_files:
    sample_file = sorted(tsv_files)[0]
    print(f"Kết quả sau Phase 2 — Qwen Verification (File: {sample_file}):\n")
    df_sample = pd.read_csv(sample_file, sep="\t")
    pd.set_option('display.max_colwidth', None)
    display(df_sample.head(25))
    nan_count = df_sample['han_sentence'].isna().sum() + df_sample['viet_sentence'].isna().sum()
    print(f"\nTổng số dòng: {len(df_sample)} | Dòng NaN còn lại (sẽ được Phase 3 xử lý): {nan_count}")
else:
    print("Chưa tìm thấy kết quả.")

### Tải về kết quả Phase 2

In [ ]:
!rm -f output_phase2.zip
!zip -q -r output_phase2.zip output/
print("Đã nén xong! Hãy tải file 'output_phase2.zip' ở cột bên phải Kaggle.")

## Bước 3.7: Phase 3 — Qwen Local Re-Alignment (Sửa NaN clusters)

Phase 3 quét các khối NaN liên tiếp còn sót lại sau Phase 2, gom chúng lại và yêu cầu Qwen **tự dóng hàng lại cục bộ** bằng cách trả về JSON.
Đây là bước tốn VRAM nhất vì Qwen phải sinh ra JSON dài, nên cần **T4 GPU**.

In [ ]:
# Chạy Phase 3: Qwen Local Re-Alignment
# --qwen bắt buộc phải bật cùng với --realign (Phase 3 chạy SAU Phase 2)
!python run_mapping.py \
    --aligner ensemble \
    --qwen \
    --realign \
    --sino_dir dataset/MAPPING/sino_extract \
    --viet_dir dataset/MAPPING/vietnam_extract/csv \
    --output_dir output \
    --work_code HVB_001 \
    --device cuda \
    --dev

### Kiểm tra kết quả sau Phase 3 (Kết quả cuối cùng)

Hãy quan sát xem các dòng NaN trước đây đã được Qwen dóng hàng lại chưa, và chất lượng có khả quan hơn Phase 2 không.

In [ ]:
tsv_files = glob.glob("output/HVB_001/**/*.tsv", recursive=True)
if tsv_files:
    sample_file = sorted(tsv_files)[0]
    print(f"Kết quả CUỐI CÙNG sau Phase 3 (File: {sample_file}):\n")
    df_final = pd.read_csv(sample_file, sep="\t")
    pd.set_option('display.max_colwidth', None)
    display(df_final.head(30))
    nan_count = df_final['han_sentence'].isna().sum() + df_final['viet_sentence'].isna().sum()
    total = len(df_final)
    matched = df_final['han_sentence'].notna() & df_final['viet_sentence'].notna()
    print(f"\nTổng số dòng: {total}")
    print(f"Dóng hàng thành công (có cả Hán và Việt): {matched.sum()} ({matched.mean()*100:.1f}%)")
    print(f"Dòng NaN còn lại (unmatched): {nan_count}")
else:
    print("Chưa tìm thấy kết quả.")

### Tải về kết quả sau cùng (Download Final Phase 3 Files)

In [ ]:
!rm -f output_final.zip
!zip -q -r output_final.zip output/
print("Đã nén xong! Hãy tải file 'output_final.zip' ở cột bên phải Kaggle để gửi lại cho mình kiểm tra.")

## Bước 4: Chuẩn bị dữ liệu huấn luyện Dịch máy

Ta chạy script `prepare_data.py` để gộp toàn bộ các file tsv song song đã được dóng hàng thành công, chia tập Train/Val (90/10) và lưu thành định dạng JSONLines chuẩn.

In [ ]:
# Tạo train.json và val.json từ kết quả dóng hàng
!python scripts/prepare_data.py

# Kiểm tra file đầu ra
!ls -la output/translation_dataset

## Bước 5: Huấn luyện mô hình dịch (Fine-tuning)

Tải script huấn luyện chuẩn từ Hugging Face và tiến hành tinh chỉnh mô hình dịch Trung-Việt.

**Lưu ý:** `--overwrite_output_dir` được viết dưới dạng cờ (flag) boolean chuẩn của HfArgumentParser.

In [ ]:
# Tải script huấn luyện chính thức
!wget -q https://raw.githubusercontent.com/huggingface/transformers/main/examples/pytorch/translation/run_translation.py

# Chạy tinh chỉnh mô hình Helsinki-NLP/opus-mt-zh-vi trên GPU
!python run_translation.py \
    --model_name_or_path Helsinki-NLP/opus-mt-zh-vi \
    --source_lang zh \
    --target_lang vi \
    --train_file output/translation_dataset/train.json \
    --validation_file output/translation_dataset/val.json \
    --output_dir ./han_viet_translation_model \
    --per_device_train_batch_size 16 \
    --per_device_eval_batch_size 16 \
    --overwrite_output_dir \
    --do_train \
    --do_eval \
    --num_train_epochs 5 \
    --learning_rate 2e-5 \
    --weight_decay 0.01 \
    --predict_with_generate \
    --eval_strategy epoch \
    --save_strategy epoch

## Bước 6: Kiểm tra khả năng dịch của Mô hình

Dịch thử câu mẫu Hán cổ bằng checkpoint vừa train.

In [ ]:
import os
from transformers import MarianMTModel, MarianTokenizer

# Sử dụng đường dẫn tuyệt đối để tránh lỗi xác thực Repo ID của Hugging Face
model_path = os.path.abspath("./han_viet_translation_model")

if os.path.exists(model_path) and os.path.exists(os.path.join(model_path, "config.json")):
    print(f"Loading model from: {model_path}")
    tokenizer = MarianTokenizer.from_pretrained(model_path)
    model = MarianMTModel.from_pretrained(model_path)

    def translate(text):
        inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True)
        translated = model.generate(**inputs)
        return tokenizer.decode(translated[0], skip_special_tokens=True)

    # Test dịch câu mẫu
    sample_han = "大南一統志卷之二承天府上"
    print(f"Hán: {sample_han}")
    print(f"Dịch: {translate(sample_han)}")
else:
    print(f"[Lưu ý] Không tìm thấy thư mục mô hình hoặc config.json tại '{model_path}'.")
    print("Vui lòng kiểm tra xem bước huấn luyện (Bước 5) đã chạy hoàn thành và lưu kết quả thành công chưa.")